# VisoMaster Internal Installation & Validation Guide

**Welcome!** This Jupyter Notebook is your interactive guide running *inside* the VisoMaster Docker container on Vast.ai.

**Purpose:**
1.  **Verify:** Run automated checks to confirm the Docker image was built correctly and essential services are running.
2.  **Troubleshoot:** Access logs and manage services (like VNC or the VisoMaster app) if things aren't working as expected.
3.  **Manual Fallback:** Provides commands to manually install dependencies or download models if the initial automated setup failed during the Docker build.

**How to Use:**
*   Read the explanations in the **Markdown cells** (like this one).
*   Run the **Code cells** (containing commands starting with `!`) by selecting the cell and pressing `Shift + Enter` or clicking the 'Run' button in the toolbar.
*   Observe the output below each code cell to see the results of the commands.

**External Documentation:** For the complete guide on setting up the GitHub repository, building the image, launching on Vast.ai, and understanding the overall pipeline, please refer to the `visomaster_deploy_documentation.md` file in your source code repository.

## 1. Automated Configuration Checks

Run the following cells to automatically verify key parts of the setup.

### 1.1 Conda Environment

In [ ]:
# Check if the 'visomaster' conda environment exists
!conda info --envs | grep visomaster

In [ ]:
# Check the active conda environment (should ideally be 'visomaster' if shell integration works, but commands below use `conda run` for safety)
!echo "Active Conda Env (might be base): $CONDA_DEFAULT_ENV"

In [ ]:
# Check Python version within the 'visomaster' environment
# Expected: Python 3.10.13 (or the version specified in Dockerfile)
!conda run -n visomaster python --version

### 1.2 CUDA & cuDNN Installation (via Conda)

In [ ]:
# Check if Conda packages for CUDA runtime and cuDNN are installed in the 'visomaster' env
# Expected: Lines showing 'cuda-runtime' (version matching 12.4.1 label) and 'cudnn'
!conda list -n visomaster | grep -E 'cuda-runtime|cudnn'

### 1.3 Python Dependencies (via Pip)

In [ ]:
# List installed pip packages in the 'visomaster' env
# Verify key packages like 'torch', 'tensorflow', 'opencv-python' etc. are present
!conda run -n visomaster pip list | head -n 20 # Show first 20 packages

In [ ]:
# Alternatively, try importing key libraries
!conda run -n visomaster python -c "import torch; print(f'PyTorch version: {torch.__version__} - Imported OK')"

In [ ]:
!conda run -n visomaster python -c "import tensorflow; print(f'TensorFlow version: {tensorflow.__version__} - Imported OK')"

### 1.4 Copied Dependencies & Models

In [ ]:
# Check if the contents of the 'dependencies' folder were copied
# Expected: List of asset files that you placed in the 'dependencies' folder before building.
!ls -lh /app/dependencies/

In [ ]:
# Check if the models directory exists and contains files/folders
# Expected: List of downloaded model files/directories.
# Note: This assumes models were downloaded to /app/models during build or via provisioning script.
!ls -lh /app/models/

### 1.5 GPU Accessibility

In [ ]:
# Check if PyTorch can detect the GPU provided by Vast.ai
# Expected: CUDA available: True
!conda run -n visomaster python -c "import torch; print(f'CUDA available: {torch.cuda.is_available()}')"

In [ ]:
# Check if TensorFlow can detect the GPU
# Expected: A list containing GPU devices, e.g., [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
!conda run -n visomaster python -c "import tensorflow as tf; print(f'TensorFlow Physical GPUs: {tf.config.list_physical_devices(\'GPU\')}')"

In [ ]:
# Check nvidia-smi command (if installed in the image, often available with CUDA toolkit)
!nvidia-smi

### 1.6 Supervisord Service Status

In [ ]:
# Check the status of services managed by Supervisor (VNC, VisoMaster app, etc.)
# Expected: All services should show state RUNNING.
!supervisorctl status

### 1.7 VNC/GUI Service Check

In [ ]:
# Check if VNC related processes are running
# Expected: Lines containing 'Xvfb', 'fluxbox', 'x11vnc'
!ps aux | grep -E 'Xvfb|fluxbox|x11vnc'

## 2. Manual Fallback Installation/Setup

If any of the checks above failed, particularly related to Conda environment or packages, you might need to run the installation steps manually. 

**IMPORTANT:** Only run these if you are sure the automated build process failed for these steps. Running them again might cause issues if the initial setup was actually successful.

**How to run:** You can run these commands directly in a Jupyter terminal. Open one via `File -> New -> Terminal` in the JupyterLab interface.

Alternatively, you can uncomment the `!` commands in the cells below and run them, but using a terminal is often easier for interactive installation.

### 2.1 Activate Environment (in Terminal)

```bash
conda activate visomaster
```

### 2.2 Install Conda CUDA/cuDNN (in Terminal after activating env)

```bash
# Make sure you are in the 'visomaster' environment first!
conda install -y -c nvidia/label/cuda-12.4.1 cuda-runtime
conda install -y -c conda-forge cudnn
conda clean -a -y
```

In [ ]:
# Or run via cell (uncomment first):
# !conda run -n visomaster conda install -y -c nvidia/label/cuda-12.4.1 cuda-runtime
# !conda run -n visomaster conda install -y -c conda-forge cudnn
# !conda run -n visomaster conda clean -a -y

### 2.3 Install Pip Dependencies (in Terminal after activating env)

```bash
# Make sure you are in the 'visomaster' environment and in the directory containing requirements.txt!
cd /app/VisoMaster 
pip install --no-cache-dir -r requirements.txt \
    --extra-index-url https://download.pytorch.org/whl/cu124 \
    --extra-index-url https://pypi.nvidia.com
```

In [ ]:
# Or run via cell (uncomment first):
# !cd /app/VisoMaster && conda run -n visomaster pip install --no-cache-dir -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu124 --extra-index-url https://pypi.nvidia.com

### 2.4 Manually Run Model Download (in Terminal after activating env)

```bash
# Make sure you are in the 'visomaster' environment!
cd /app/VisoMaster
python download_models.py --output_dir /app/models
```

In [ ]:
# Or run via cell (uncomment first):
# !cd /app/VisoMaster && conda run -n visomaster python download_models.py --output_dir /app/models

### 2.5 Restart Services After Manual Changes

If you made manual changes, you might need to restart the affected services using `supervisorctl` (see Section 3).

## 3. Troubleshooting & Service Management

Use `supervisorctl` to check status, stop, start, or restart services.

In [ ]:
# View status of all services
!supervisorctl status

In [ ]:
# Example: Restart the VisoMaster application
# Replace 'visomaster_app' with the correct service name from `supervisorctl status` if different.
!supervisorctl restart visomaster_app

In [ ]:
# Example: Restart the VNC server
# Replace 'x11vnc' with the correct service name if different.
!supervisorctl restart x11vnc

**Common Issues & Tips:**
*   **Service in `FATAL` or `EXITED` state:** Check its logs (see Section 4) for error messages.
*   **VNC connection refused:** Ensure `x11vnc` (or your VNC server) is `RUNNING`. Check its logs. Ensure the port mapping on Vast.ai is correct (Host Port -> 5901).
*   **VisoMaster App crashing:** Check `visomaster_app` logs for Python errors. Ensure dependencies are installed correctly (Section 1) and models are downloaded (Section 1.4). Verify GPU access (Section 1.5).
*   **`supervisorctl` command not found:** Supervisor might not be running correctly. Check `/var/log/supervisor/supervisord.log` (see Section 4).
*   **Permission errors:** Files in `/workspace` might have incorrect permissions if created outside the container with different user IDs. Use `chmod` or `chown` in a terminal if necessary.

## 4. Log Viewer

Access logs to diagnose issues. Use `cat` to view the whole file, `tail` to view the end, and `tail -f` to follow logs in real-time (run `tail -f` in a Jupyter terminal).

### 4.1 Supervisord Logs

In [ ]:
# Main supervisord log (for startup issues)
!tail -n 50 /var/log/supervisor/supervisord.log

In [ ]:
# VisoMaster App standard output log
!tail -n 100 /var/log/supervisor/visomaster_app.log

In [ ]:
# VisoMaster App standard error log (IMPORTANT for errors!)
!tail -n 100 /var/log/supervisor/visomaster_app_err.log

In [ ]:
# X11VNC standard output log
!tail -n 100 /var/log/supervisor/x11vnc.log

In [ ]:
# X11VNC standard error log
!tail -n 100 /var/log/supervisor/x11vnc_err.log

### 4.2 Vast.ai Instance Portal Logs (If Portal Alternative is Used)

If you deployed using the Instance Portal alternative configuration, logs are often redirected to `/var/log/portal/`. Check the `visomaster_deploy_supervisord_alt_portal.conf` file to confirm log paths.

In [ ]:
# Example: Caddy log (if portal is used)
# !tail -n 100 /var/log/portal/caddy.log

In [ ]:
# Example: VisoMaster App log (if portal is used)
# !tail -n 100 /var/log/portal/visomaster_app.log

### 4.3 Following Logs (Real-time)

To follow logs in real-time, open a **Jupyter Terminal** (`File -> New -> Terminal`) and run:

```bash
# Example: Follow VisoMaster app error log
tail -f /var/log/supervisor/visomaster_app_err.log

# Example: Follow VNC error log
tail -f /var/log/supervisor/x11vnc_err.log

# Press Ctrl+C in the terminal to stop following
```